[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C47_RecSys_Ranking_Course/02_matrix_factorization/02_matrix_factorization.ipynb)

# 02 · 矩阵分解（用 numpy 从零）

目标：从零实现**带偏置的矩阵分解（FunkSVD, SGD 训练）**——看 **RMSE 收敛**、**数值梯度对拍解析梯度**、对比**有无偏置**、实现**一步 ALS** 并验证其最优性。

路线：复用模块 00 数据 → 预测与损失 → 解析梯度 vs 数值梯度对拍 → SGD 训练看收敛 → 偏置消融 → ALS 闭式解 → ✏️ 练习 → 📖 答案 → 🧪 真实 MovieLens RMSE 胶囊。

> 核心心智：**给每个用户/物品学一个 k 维向量，内积≈评分；在观测项上最小化(误差²+正则)，SGD 或 ALS 求解。**

## 0 · 数据基座 + 训练/测试切分

复用模块 00 加载器，按观测随机切 80/20 训练/测试（评分预测任务）。

In [ ]:
import numpy as np

def load_movielens_or_synth(n_users=200, n_items=300, rank=8, seed=0, verbose=True):
    import os
    for path in ['ml-100k/u.data', 'u.data', os.path.expanduser('~/ml-100k/u.data')]:
        if os.path.exists(path):
            d = np.loadtxt(path, dtype=np.int64)[:, :3].astype(float)
            d[:,0]-=1; d[:,1]-=1
            if verbose: print(f'真实 MovieLens-100k: {len(d)} 评分')
            return d, int(d[:,0].max())+1, int(d[:,1].max())+1
    try:
        import urllib.request
        raw = urllib.request.urlopen('https://files.grouplens.org/datasets/movielens/ml-100k/u.data', timeout=5).read().decode()
        rows = [list(map(int, ln.split('\t')[:3])) for ln in raw.strip().split('\n')]
        d = np.array(rows, dtype=float); d[:,0]-=1; d[:,1]-=1
        return d, int(d[:,0].max())+1, int(d[:,1].max())+1
    except Exception as e:
        if verbose: print(f'回退合成（{type(e).__name__}）')
    rng = np.random.default_rng(seed)
    P = rng.standard_normal((n_users, rank))*0.5; Q = rng.standard_normal((n_items, rank))*0.5
    bu = rng.standard_normal(n_users)*0.3; bi = rng.standard_normal(n_items)*0.5
    rows = []
    for u in range(n_users):
        for i in rng.choice(n_items, size=rng.integers(20,60), replace=False):
            r = 3.5 + bu[u] + bi[i] + P[u]@Q[i] + rng.standard_normal()*0.3
            rows.append([u, i, float(np.clip(np.round(r*2)/2, 1, 5))])
    d = np.array(rows, dtype=float)
    if verbose: print(f'合成评分: {len(d)} 条')
    return d, n_users, n_items

ratings, n_users, n_items = load_movielens_or_synth(seed=0)
rng = np.random.default_rng(42)
perm = rng.permutation(len(ratings))
split = int(0.8*len(ratings))
train, test = ratings[perm[:split]], ratings[perm[split:]]
print(f'{n_users} 用户, {n_items} 物品 | 训练 {len(train)}, 测试 {len(test)}')
assert len(train)+len(test) == len(ratings)
print('✅ 数据与切分就绪')

## 1 · 预测与损失

带偏置预测：$\hat r_{ui}=\mu+b_u+b_i+p_u\cdot q_i$。损失 = 观测项上的平方误差 + L2 正则。
先把模型参数和这两个函数立起来。

In [ ]:
def init_mf(n_users, n_items, k=16, seed=0):
    rng = np.random.default_rng(seed)
    return {
        'P': rng.standard_normal((n_users, k))*0.1,
        'Q': rng.standard_normal((n_items, k))*0.1,
        'bu': np.zeros(n_users), 'bi': np.zeros(n_items),
        'mu': 0.0, 'k': k,
    }

def predict(m, u, i):
    return m['mu'] + m['bu'][u] + m['bi'][i] + m['P'][u] @ m['Q'][i]

def predict_batch(m, users, items):
    users = users.astype(int); items = items.astype(int)
    dot = np.sum(m['P'][users] * m['Q'][items], axis=1)
    return m['mu'] + m['bu'][users] + m['bi'][items] + dot

def loss(m, data, lam):
    pred = predict_batch(m, data[:,0], data[:,1])
    err = data[:,2] - pred
    reg = lam*(np.sum(m['P']**2) + np.sum(m['Q']**2) + np.sum(m['bu']**2) + np.sum(m['bi']**2))
    return float(np.sum(err**2) + reg)

def rmse(m, data):
    pred = predict_batch(m, data[:,0], data[:,1])
    return float(np.sqrt(np.mean((data[:,2]-pred)**2)))

m = init_mf(n_users, n_items, k=16)
m['mu'] = train[:,2].mean()                          # 全局均值初始化 mu
print(f'初始(仅mu) 训练RMSE={rmse(m, train):.4f}, 测试RMSE={rmse(m, test):.4f}')
# 只用 mu 预测，RMSE 应约等于评分的标准差
assert abs(rmse(m, train) - train[:,2].std()) < 0.05
print('✅ 预测/损失就位；仅用全局均值时 RMSE ≈ 评分标准差（符合预期）')

## 2 · 解析梯度 vs 数值梯度（对拍）

SGD 的命脉是梯度对不对。单点误差 $e=r-\hat r$，梯度：
$$\nabla_{p_u}\ell = -2e\,q_i + 2\lambda p_u,\quad \nabla_{q_i}\ell = -2e\,p_u+2\lambda q_i,\quad \partial_{b_u}\ell=-2e+2\lambda b_u$$

**用数值梯度（有限差分）对拍解析梯度**——这是验证梯度推导正确的黄金标准。

In [ ]:
def grad_point(m, u, i, r, lam):
    '''单点 (u,i,r) 的解析梯度（含正则）。返回各参数的梯度。'''
    e = r - predict(m, u, i)
    g_pu = -2*e*m['Q'][i] + 2*lam*m['P'][u]
    g_qi = -2*e*m['P'][u] + 2*lam*m['Q'][i]
    g_bu = -2*e + 2*lam*m['bu'][u]
    g_bi = -2*e + 2*lam*m['bi'][i]
    return g_pu, g_qi, g_bu, g_bi

def point_loss(m, u, i, r, lam):
    e = r - predict(m, u, i)
    return e**2 + lam*(m['P'][u]@m['P'][u] + m['Q'][i]@m['Q'][i] + m['bu'][u]**2 + m['bi'][i]**2)

# 随机初始化一个非零模型来测梯度
m = init_mf(n_users, n_items, k=8, seed=3)
m['bu'] = np.random.default_rng(1).standard_normal(n_users)*0.1
m['bi'] = np.random.default_rng(2).standard_normal(n_items)*0.1
u, i, r, lam = 5, 7, 4.0, 0.05
g_pu, g_qi, g_bu, g_bi = grad_point(m, u, i, r, lam)

eps = 1e-6
# 数值梯度 for p_u[0]
m['P'][u,0] += eps; lp = point_loss(m, u, i, r, lam)
m['P'][u,0] -= 2*eps; lm = point_loss(m, u, i, r, lam)
m['P'][u,0] += eps
num_pu0 = (lp - lm)/(2*eps)
# 数值梯度 for b_u
m['bu'][u] += eps; lp = point_loss(m, u, i, r, lam)
m['bu'][u] -= 2*eps; lm = point_loss(m, u, i, r, lam)
m['bu'][u] += eps
num_bu = (lp - lm)/(2*eps)
print(f'p_u[0]: 解析={g_pu[0]:.6f} 数值={num_pu0:.6f}')
print(f'b_u   : 解析={g_bu:.6f} 数值={num_bu:.6f}')
assert abs(g_pu[0]-num_pu0) < 1e-4, '解析梯度 p_u 与数值不符'
assert abs(g_bu-num_bu) < 1e-4, '解析梯度 b_u 与数值不符'
print('✅ 解析梯度对拍数值梯度通过——梯度推导正确，可以放心 SGD')

## 3 · SGD 训练：看 RMSE 单调收敛

更新规则（吸收常数 2 进 $\eta$）：$p_u\mathrel{+}=\eta(e\,q_i-\lambda p_u)$，$q_i\mathrel{+}=\eta(e\,p_u-\lambda q_i)$，偏置同理。
**关键：先算完 $e$ 和旧的 $p_u,q_i$，再同时更新**（不能用更新后的 $p_u$ 去算 $q_i$）。

训练若正确，**训练 RMSE 应逐 epoch 单调下降**并收敛。

In [ ]:
def train_sgd(train, n_users, n_items, k=16, lr=0.01, lam=0.05, epochs=15, seed=0, verbose=False):
    m = init_mf(n_users, n_items, k, seed)
    m['mu'] = train[:,2].mean()
    rng = np.random.default_rng(seed)
    history = []
    for ep in range(epochs):
        order = rng.permutation(len(train))
        for t in order:
            u, i = int(train[t,0]), int(train[t,1]); r = train[t,2]
            e = r - predict(m, u, i)
            pu_old = m['P'][u].copy()                 # 先存旧值
            m['P'][u] += lr*(e*m['Q'][i] - lam*m['P'][u])
            m['Q'][i] += lr*(e*pu_old   - lam*m['Q'][i])  # 用旧 pu！
            m['bu'][u] += lr*(e - lam*m['bu'][u])
            m['bi'][i] += lr*(e - lam*m['bi'][i])
        tr_rmse = rmse(m, train)
        history.append(tr_rmse)
        if verbose: print(f'  epoch {ep+1:2d}: train RMSE={tr_rmse:.4f}')
    return m, history

m, hist = train_sgd(train, n_users, n_items, k=16, lr=0.01, lam=0.05, epochs=15, verbose=True)
print(f'\n最终 训练RMSE={rmse(m, train):.4f}, 测试RMSE={rmse(m, test):.4f}')
# 收敛性：训练 RMSE 应（基本）单调下降
drops = sum(hist[t+1] <= hist[t] + 1e-6 for t in range(len(hist)-1))
assert drops >= len(hist)-3, f'训练RMSE应基本单调下降, 只有 {drops}/{len(hist)-1} 步下降'
assert hist[-1] < hist[0], 'RMSE 应明显下降'
assert rmse(m, test) < test[:,2].std(), 'MF 测试 RMSE 应优于「只猜均值」'
print('✅ SGD 收敛：训练 RMSE 单调下降，测试 RMSE 优于均值基线')

## 4 · 偏置消融：偏置贡献了多少？

Koren 2009 强调偏置项贡献巨大。验证：① 只用 $\mu+b_u+b_i$（无内积，`k=0` 思想，这里冻结 P,Q=0）；② 完整模型。对比测试 RMSE。

In [ ]:
def train_bias_only(train, n_users, n_items, lr=0.01, lam=0.05, epochs=15, seed=0):
    '''只学偏置 mu+bu+bi，内积部分恒为 0。'''
    m = init_mf(n_users, n_items, k=1, seed=seed)
    m['P'][:] = 0; m['Q'][:] = 0                       # 冻结内积为 0
    m['mu'] = train[:,2].mean()
    rng = np.random.default_rng(seed)
    for ep in range(epochs):
        for t in rng.permutation(len(train)):
            u, i = int(train[t,0]), int(train[t,1]); r = train[t,2]
            e = r - (m['mu'] + m['bu'][u] + m['bi'][i])
            m['bu'][u] += lr*(e - lam*m['bu'][u])
            m['bi'][i] += lr*(e - lam*m['bi'][i])
    return m

m_bias = train_bias_only(train, n_users, n_items, epochs=15)
rmse_meanonly = test[:,2].std()                        # 仅全局均值
rmse_bias = rmse(m_bias, test)                          # +偏置
rmse_full = rmse(m, test)                               # 完整(上一节训好的 m)
print(f'仅全局均值     测试RMSE = {rmse_meanonly:.4f}')
print(f'+用户/物品偏置 测试RMSE = {rmse_bias:.4f}  (偏置带来的提升)')
print(f'+隐因子内积    测试RMSE = {rmse_full:.4f}  (完整模型)')
assert rmse_bias < rmse_meanonly, '偏置应优于纯均值'
assert rmse_full < rmse_bias, '加内积应进一步提升'
print('✅ 验证 Koren 的观点：光偏置就显著降 RMSE，内积再添一截。先吃干净偏置，内积捕捉个性化。')

## 5 · ALS：用户因子的闭式解

固定物品因子 $Q$，单个用户的最优 $p_u$ 有闭式解（岭回归）：
$$p_u = (Q_u^\top Q_u + \lambda I)^{-1} Q_u^\top r_u$$
其中 $Q_u$ 是 $u$ 评过物品的因子堆叠、$r_u$ 对应评分（这里用去偏后的残差）。

验证：闭式解确实让该用户的局部目标达到最小（比解前的 $p_u$ 损失更低，且梯度≈0）。

In [ ]:
def als_user_step(m, u, train_by_user, lam):
    '''固定 Q,偏置, 闭式解用户 u 的 p_u（拟合去偏残差）。'''
    items = train_by_user[u]['items']
    if len(items) == 0:
        return m['P'][u]
    Qu = m['Q'][items]                                  # (n_u, k)
    resid = train_by_user[u]['ratings'] - m['mu'] - m['bu'][u] - m['bi'][items]
    A = Qu.T @ Qu + lam*np.eye(m['k'])                  # (k,k)
    b = Qu.T @ resid                                    # (k,)
    return np.linalg.solve(A, b)

# 建用户索引
def index_by_user(train, n_users):
    by = [{'items': [], 'ratings': []} for _ in range(n_users)]
    for t in range(len(train)):
        u = int(train[t,0]); by[u]['items'].append(int(train[t,1])); by[u]['ratings'].append(train[t,2])
    for u in range(n_users):
        by[u]['items'] = np.array(by[u]['items'], dtype=int)
        by[u]['ratings'] = np.array(by[u]['ratings'])
    return by

tbu = index_by_user(train, n_users)
# 取一个评分较多的用户
u = int(np.argmax([len(tbu[u]['items']) for u in range(n_users)]))
lam = 0.1

def user_local_loss(m, u):
    items = tbu[u]['items']
    pred = m['mu'] + m['bu'][u] + m['bi'][items] + m['Q'][items] @ m['P'][u]
    return np.sum((tbu[u]['ratings']-pred)**2) + lam*(m['P'][u]@m['P'][u])

loss_before = user_local_loss(m, u)
p_new = als_user_step(m, u, tbu, lam)
m['P'][u] = p_new
loss_after = user_local_loss(m, u)
print(f'用户 {u}（{len(tbu[u]["items"])} 条评分）局部损失: 解前={loss_before:.3f} -> 解后={loss_after:.3f}')
assert loss_after <= loss_before + 1e-6, 'ALS 闭式解应使局部损失不增'
# 验证最优性：在最优点对 p_u 的梯度应≈0
items = tbu[u]['items']; Qu = m['Q'][items]
resid = tbu[u]['ratings'] - m['mu'] - m['bu'][u] - m['bi'][items]
grad = -2*Qu.T@(resid - Qu@m['P'][u]) + 2*lam*m['P'][u]
assert np.linalg.norm(grad) < 1e-8, f'最优点梯度应≈0, 得到 {np.linalg.norm(grad):.2e}'
print('✅ ALS 闭式解正确：局部损失最小化、梯度≈0（这是 ALS「每步精确求解」的体现）')

## 6 · 从评分预测到 Top-N 推荐

RMSE 是评分预测指标，但工业要的是 **Top-N 排序**。MF 学好后，对用户 $u$ 给所有未评物品打分 $\mu+b_u+b_i+p_u\cdot q_i$，取最高的 N 个。
下面用训好的 MF 生成 Top-N，并验证它能把「用户实际高分的物品」排进前列（leave-one-out Recall）。

In [ ]:
def mf_topn(m, u, seen_items, n=10):
    '''MF 对用户 u 的所有未评物品打分，返回 Top-N 物品 id。'''
    scores = m['mu'] + m['bu'][u] + m['bi'] + m['Q'] @ m['P'][u]   # 对全部物品打分
    scores = scores.copy()
    for i in seen_items: scores[i] = -np.inf                       # 屏蔽已评
    return np.argsort(-scores)[:n]

# 训一个 MF，用留一评估 Top-N
m_topn, _ = train_sgd(train, n_users, n_items, k=32, lr=0.01, lam=0.05, epochs=18)
seen = {}
for t in range(len(train)):
    seen.setdefault(int(train[t,0]), set()).add(int(train[t,1]))
# 用测试集里高分(>=4)的物品当「相关」，看是否进 Top-20
rng_e = np.random.default_rng(0); hits = 0; tries = 0
test_high = test[test[:,2] >= 4.0]
for t in rng_e.choice(len(test_high), size=min(100, len(test_high)), replace=False):
    u, held = int(test_high[t,0]), int(test_high[t,1])
    topn = mf_topn(m_topn, u, seen.get(u, set()), n=20)
    hits += int(held in topn); tries += 1
recall = hits/tries
print(f'MF Top-20 Recall (高分测试物品命中率) = {recall:.4f} (随机 ≈ {20/n_items:.4f})')
assert recall > 20/n_items, 'MF Top-N 应远超随机'
print('✅ 同一个 MF 既能预测评分(RMSE)，也能做 Top-N 排序——隐因子是通用的表示')

---
## ✏️ 练习 1：实现单点 SGD 更新

实现 `sgd_step`：对单个观测 $(u,i,r)$ 做一步 SGD 更新（**原地修改** `m`）。
记住更新规则与「先存旧 $p_u$」的陷阱。

In [ ]:
def sgd_step(m, u, i, r, lr, lam):
    '''对单点 (u,i,r) 原地更新 m 的 P[u], Q[i], bu[u], bi[i]。'''
    # TODO:
    #  1) e = r - predict(m, u, i)
    #  2) 先存 pu_old = m['P'][u].copy()
    #  3) m['P'][u] += lr*(e*m['Q'][i] - lam*m['P'][u])
    #     m['Q'][i] += lr*(e*pu_old   - lam*m['Q'][i])   # 用 pu_old
    #  4) 更新 bu[u], bi[i]: += lr*(e - lam*b)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
m = init_mf(n_users, n_items, k=8, seed=5); m['mu'] = train[:,2].mean()
rng = np.random.default_rng(0)
l0 = loss(m, train, lam=0.05)
for _ in range(3):
    for t in rng.permutation(len(train)):
        sgd_step(m, int(train[t,0]), int(train[t,1]), train[t,2], lr=0.01, lam=0.05)
l1 = loss(m, train, lam=0.05)
print(f'训练损失: {l0:.1f} -> {l1:.1f}')
assert l1 < l0, 'SGD 更新应降低训练损失'
# 对拍：一步 sgd_step 应等于手算
m2 = init_mf(n_users, n_items, k=8, seed=9)
u,i,r = 3,4,5.0; e = r - predict(m2,u,i); pu_old=m2['P'][u].copy()
expect_pu = m2['P'][u] + 0.02*(e*m2['Q'][i]-0.1*m2['P'][u])
sgd_step(m2, u, i, r, lr=0.02, lam=0.1)
assert np.allclose(m2['P'][u], expect_pu, atol=1e-12), '单步更新与手算应一致'
print('✅ 练习 1 通过：SGD 单步更新正确（且降损失、对拍手算）')

## ✏️ 练习 2：ALS 物品因子的闭式解

对称于用户更新，实现 `als_item_step`：固定 $P$、偏置，闭式解物品 $i$ 的 $q_i$。
$$q_i = (P_i^\top P_i + \lambda I)^{-1} P_i^\top \text{resid}_i$$
其中 $P_i$ 是评过 $i$ 的用户因子堆叠，$\text{resid}_i$ 是去偏残差。

In [ ]:
def als_item_step(m, i, train_by_item, lam):
    '''固定 P,偏置, 闭式解物品 i 的 q_i。train_by_item[i] = {'users':..., 'ratings':...}'''
    users = train_by_item[i]['users']
    if len(users) == 0:
        return m['Q'][i]
    # TODO: Pi = m['P'][users]; resid = ratings - mu - bu[users] - bi[i]
    #       A = Pi.T@Pi + lam*I; b = Pi.T@resid; 返回 solve(A,b)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
def index_by_item(train, n_items):
    by = [{'users': [], 'ratings': []} for _ in range(n_items)]
    for t in range(len(train)):
        i = int(train[t,1]); by[i]['users'].append(int(train[t,0])); by[i]['ratings'].append(train[t,2])
    for i in range(n_items):
        by[i]['users'] = np.array(by[i]['users'], dtype=int); by[i]['ratings'] = np.array(by[i]['ratings'])
    return by

tbi = index_by_item(train, n_items)
i = int(np.argmax([len(tbi[i]['users']) for i in range(n_items)]))
lam = 0.1
def item_local_loss(m, i):
    users = tbi[i]['users']
    pred = m['mu'] + m['bu'][users] + m['bi'][i] + m['P'][users] @ m['Q'][i]
    return np.sum((tbi[i]['ratings']-pred)**2) + lam*(m['Q'][i]@m['Q'][i])
before = item_local_loss(m, i)
m['Q'][i] = als_item_step(m, i, tbi, lam)
after = item_local_loss(m, i)
assert after <= before + 1e-6, 'ALS 物品步应使局部损失不增'
users = tbi[i]['users']; Pi = m['P'][users]
resid = tbi[i]['ratings'] - m['mu'] - m['bu'][users] - m['bi'][i]
grad = -2*Pi.T@(resid - Pi@m['Q'][i]) + 2*lam*m['Q'][i]
assert np.linalg.norm(grad) < 1e-8, '最优点梯度应≈0'
print('✅ 练习 2 通过：ALS 物品闭式解正确（局部损失最小、梯度≈0）')

## ✏️ 练习 3：正则强度 λ 的影响

正则太小→过拟合（训练好测试差），太大→欠拟合（都差）。实现 `train_and_eval`，
对给定 λ 训练并返回 (训练RMSE, 测试RMSE)，验证存在一个最优 λ。

In [ ]:
def train_and_eval(train, test, n_users, n_items, lam, k=16, lr=0.01, epochs=12, seed=0):
    '''训练 MF 并返回 (train_rmse, test_rmse)。'''
    # TODO: 调用 train_sgd(...) 训练，返回训练/测试 RMSE
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
results = {}
for lam in [0.001, 0.05, 1.0]:
    tr, te = train_and_eval(train, test, n_users, n_items, lam=lam, epochs=12)
    results[lam] = (tr, te)
    print(f'λ={lam:6.3f}: 训练RMSE={tr:.4f}, 测试RMSE={te:.4f}, 过拟合gap={te-tr:.4f}')
# 小 λ 过拟合：训练-测试 gap 应比大 λ 大
gap_small = results[0.001][1] - results[0.001][0]
gap_large = results[1.0][1] - results[1.0][0]
assert gap_small > gap_large, '小 λ 的过拟合 gap 应更大'
# 大 λ 欠拟合：训练 RMSE 应比中等 λ 差
assert results[1.0][0] > results[0.05][0], '过强正则应欠拟合（训练RMSE更差）'
print('✅ 练习 3 通过：观察到过拟合(小λ)与欠拟合(大λ)，中间存在最优')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def sgd_step(m, u, i, r, lr, lam):
    e = r - predict(m, u, i)
    pu_old = m['P'][u].copy()
    m['P'][u] += lr*(e*m['Q'][i] - lam*m['P'][u])
    m['Q'][i] += lr*(e*pu_old   - lam*m['Q'][i])
    m['bu'][u] += lr*(e - lam*m['bu'][u])
    m['bi'][i] += lr*(e - lam*m['bi'][i])

# 练习 2 参考答案
def als_item_step(m, i, train_by_item, lam):
    users = train_by_item[i]['users']
    if len(users) == 0:
        return m['Q'][i]
    Pi = m['P'][users]
    resid = train_by_item[i]['ratings'] - m['mu'] - m['bu'][users] - m['bi'][i]
    A = Pi.T @ Pi + lam*np.eye(m['k'])
    return np.linalg.solve(A, Pi.T @ resid)

# 练习 3 参考答案
def train_and_eval(train, test, n_users, n_items, lam, k=16, lr=0.01, epochs=12, seed=0):
    m, _ = train_sgd(train, n_users, n_items, k=k, lr=lr, lam=lam, epochs=epochs, seed=seed)
    return rmse(m, train), rmse(m, test)
print('参考答案已载入')

---
## 🧪 真实数据胶囊：MF 在 MovieLens 上的 RMSE 与「相似电影」

在真实/合成 MovieLens 上训练完整 MF，做两件工业上真实的事：① 报告测试 RMSE（对标 Netflix Prize 的指标）；
② 用学到的**物品因子内积** $q_i\cdot q_j$ 找相似电影——这是 MF 版的「看了又看」，和模块 01 的余弦相似度对比。

**TODO**：补全 `mf_similar_items`，用物品因子的**余弦相似度**返回与给定电影最相似的 topn 个。

In [ ]:
def mf_similar_items(Q, item_id, topn=5):
    '''用物品隐因子 Q 的余弦相似度找最相似的 topn 个物品（排除自己）。'''
    qn = Q / (np.linalg.norm(Q, axis=1, keepdims=True) + 1e-12)
    sims = qn @ qn[item_id]
    sims[item_id] = -np.inf
    # TODO: 返回 [(item_id, sim), ...] 按相似度降序取 topn
    raise NotImplementedError

In [ ]:
# 自测（胶囊）
m_full, _ = train_sgd(train, n_users, n_items, k=32, lr=0.01, lam=0.05, epochs=20)
test_rmse = rmse(m_full, test)
print(f'MF(k=32) 测试 RMSE = {test_rmse:.4f}')
print('（真实 MovieLens-100k 上调好的 MF 通常能到 ~0.91-0.93；合成数据因结构更简单会更低）')
neighbors = mf_similar_items(m_full['Q'].copy(), item_id=0, topn=5)
print('与电影 0 隐因子最相似的 5 部:', [(iid, round(s,3)) for iid, s in neighbors])
assert test_rmse < test[:,2].std(), 'MF 应优于均值基线'
assert len(neighbors) == 5 and all(iid != 0 for iid, _ in neighbors)
assert neighbors[0][1] >= neighbors[-1][1]
print('✅ 胶囊通过：MF 给出可用的 RMSE，且物品因子内积给出「学出来的」相似度（比手工余弦更能泛化）')

In [ ]:
# 📖 胶囊参考答案
def mf_similar_items(Q, item_id, topn=5):
    qn = Q / (np.linalg.norm(Q, axis=1, keepdims=True) + 1e-12)
    sims = qn @ qn[item_id]
    sims[item_id] = -np.inf
    order = np.argsort(-sims)[:topn]
    return [(int(o), float(sims[o])) for o in order]

### 小结
- **矩阵分解** = 给每个用户/物品学一个 k 维隐因子，内积≈评分；$R\approx PQ^\top$，低秩近似。
- 推荐里的「SVD」**不是**线代 SVD（矩阵有缺失），而是**只在观测项上**优化的 FunkSVD。
- 目标 = 平方误差 + **L2 正则**（= 高斯先验）；**SGD**（简单、在线）或 **ALS**（闭式解、并行、隐式反馈）求解。
- **偏置项** $\mu+b_u+b_i$ 贡献巨大，先吃干净偏置内积再捕捉个性化；**先把 MF 基线调好**（Rendle 警示）。

下一站：**模块 03 · 双塔召回** —— 把「embedding 查找表」升级成神经网络塔，用 in-batch 负采样训练，做亿级召回。